In [1]:
import pandas as pd
import numpy as np
import os
import math
import warnings
import pickle
import hashlib
from collections import Counter
from itertools import islice

import scipy
from scipy import stats
from scipy.stats import linregress, pearsonr, percentileofscore

import statsmodels.api as sm
import seaborn as sns
import matplotlib.pyplot as plt
import shap

from tqdm.notebook import tqdm
from mlxtend.feature_selection import SequentialFeatureSelector as SFS

from sklearn import metrics
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import KFold, GridSearchCV, train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import MinMaxScaler
from sklearn.utils import check_random_state, resample

from skopt import BayesSearchCV
from skopt.space import Real, Integer

warnings.filterwarnings("ignore")

c:\Users\eguen\miniconda3\envs\redlat_gap\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Functions

In [2]:
def get_best_features_sfs(data, vars_list, target_col="Age",
                          pkl_path="best_features_w-Barthel-Cognition_v01.pkl",
                          force=True):
    if (not force) and os.path.exists(pkl_path):
        with open(pkl_path, "rb") as f:
            best_features = pickle.load(f)
        print("Loaded best_features from pickle:", best_features)
    else:

        gbr = HistGradientBoostingRegressor()

        # Apply forward SFS using R2 as metric
        sfs = SFS(gbr,
                  k_features='best',  # or a fixed number like 5 or 10 if preferred
                  forward=True,
                  floating=False,
                  scoring='r2',
                  cv=5,
                  n_jobs=-1,
                  verbose=2)

        # Execute SFS
        sfs = sfs.fit(data[vars_list], data[target_col])

        # Get the best selected features
        best_features = list(sfs.k_feature_names_)
        with open(pkl_path, "wb") as f:
            pickle.dump(best_features, f)
        print("Computed and saved best_features:", best_features)

    return best_features

In [3]:
from sklearn.pipeline import Pipeline
def run_nested_cv_hgbr(data_, best_features,
                       y_col="Age", diag_col="anydem"):

    # Variables
    y = data_[y_col]
    X_selected = data_[best_features]  # selected variables

    # Outer CV
    kf = KFold(n_splits=10, shuffle=True, random_state=42)

    # Hyperparameters for Gradient Boosting
    param_grid = {
        "model__max_iter": [300, 400, 500, 600],
        "model__max_depth": [3, 5, 7],
        "model__learning_rate": [0.01, 0.05, 0.1]
    }

    # Results
    r2_scores = []
    r2_adj_scores = []  # new
    y_true_all = []
    y_pred_all = []
    results_labels_df = pd.DataFrame(columns=['y_labels', 'y_pred', 'GAP', 'GAP_corrected', 'ID'])

    # SHAP accumulator
    shap_values_sum = pd.Series(0, index=X_selected.columns)
    perm_values_sum = pd.Series(0, index=X_selected.columns)

    p = X_selected.shape[1]  # number of predictors (constant)

    # Nested CV
    for fold, (train_idx, test_idx) in enumerate(kf.split(X_selected)):
        print(f" Fold {fold + 1} (Nested CV)")

        X_train, X_test = X_selected.iloc[train_idx], X_selected.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # Pipeline
        pipeline = Pipeline([
            ("scaler", MinMaxScaler((0.05, 0.95))),
            ("model", HistGradientBoostingRegressor(random_state=42))
        ])

        # Inner GridSearch
        grid_search = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grid,
            scoring="r2",
            cv=5,
            n_jobs=-1,
            verbose=0
        )

        grid_search.fit(X_train, y_train)
        best_model = grid_search.best_estimator_

        # Evaluate
        y_pred = best_model.predict(X_test)
        r2 = r2_score(y_test, y_pred)
        r2_scores.append(r2)

        # Adjusted R2 per fold
        n_test = len(y_test)
        if (n_test - p - 1) > 0:
            r2_adj = 1 - (1 - r2) * (n_test - 1) / (n_test - p - 1)
        else:
            r2_adj = np.nan
        r2_adj_scores.append(r2_adj)

        print(f" R2 fold {fold + 1}: {r2:.4f} | R2 adj: {r2_adj:.4f} (best hyperparameters: {grid_search.best_params_})")

        y_true_all.extend(y_test)
        y_pred_all.extend(y_pred)

        # GAP
        gap_test = y_pred - y_test
        gap_train_all = best_model.predict(X_train) - y_train

        # Filter only CN subjects in train for regression
        train_ids = X_train.index
        diag_train = data_.loc[train_ids, diag_col]
        cn_mask = diag_train == 0

        slope, intercept, _, _, _ = linregress(y_train[cn_mask], gap_train_all[cn_mask])
        corrected_gap = gap_test - (slope * y_test + intercept)

        # SHAP values for this fold
        explainer = shap.Explainer(best_model.named_steps['model'], X_train)
        shap_values = explainer(X_test, check_additivity=False)
        shap_values_mean = np.abs(shap_values.values).mean(axis=0)
        shap_values_sum += pd.Series(shap_values_mean, index=X_selected.columns)

        perm_importance = permutation_importance(
            best_model, X_test, y_test, n_repeats=30, random_state=42, n_jobs=-1
        )

        # Average of the decrease in R2
        perm_values_mean = perm_importance.importances_mean
        perm_values_sum += pd.Series(perm_values_mean, index=X_selected.columns)

        # Save results for this fold
        result = np.column_stack((y_test, y_pred, gap_test, corrected_gap))
        temp_df = pd.DataFrame(result, columns=['y_labels', 'y_pred', 'GAP', 'GAP_corrected'])
        temp_df['ID'] = X_test.index
        results_labels_df = pd.concat([results_labels_df, temp_df], ignore_index=True)

    # r (Pearson correlation)
    r = np.corrcoef(y_true_all, y_pred_all)[0, 1]

    # RMSE
    rmse = np.sqrt(mean_squared_error(y_true_all, y_pred_all))

    # MAE
    mae = mean_absolute_error(y_true_all, y_pred_all)

    # Cohen's f2: f2 = R2 / (1 - R2)
    f2 = r2 / (1 - r2) if r2 < 1 else np.inf

    # Global R2 and global adjusted R2
    r2_global = r2_score(y_true_all, y_pred_all)
    n_global = len(y_true_all)
    if (n_global - p - 1) > 0:
        r2_adj_global = 1 - (1 - r2_global) * (n_global - 1) / (n_global - p - 1)
    else:
        r2_adj_global = np.nan

    # Final results
    print("\n R2 per fold (Nested CV):", r2_scores)
    print(" R2 adjusted per fold (Nested CV):", r2_adj_scores)
    print(" Average R2:", np.mean(r2_scores), np.std(r2_scores))
    print(" Average adjusted R2:", np.nanmean(r2_adj_scores), np.nanstd(r2_adj_scores))
    print(" Global R2 (all predictions):", r2_global)
    print(" Global adjusted R2:", r2_adj_global)

    print(" Cohen's f2:", f2)
    print(" r (correlation):", r)
    print(" RMSE:", rmse)
    print(" MAE:", mae)

    # Average SHAP importance
    shap_importance_avg = shap_values_sum / kf.get_n_splits()
    shap_importance_avg = shap_importance_avg.sort_values(ascending=False)
    print("\n Average SHAP values importance:")
    print(shap_importance_avg)

    # Average permutation importance
    perm_importance_avg = perm_values_sum / kf.get_n_splits()
    perm_importance_avg = perm_importance_avg.sort_values(ascending=False)
    print("\n Average importance (Permutation Importance):")
    print(perm_importance_avg)

    return (r2_scores, r2_adj_scores, results_labels_df, shap_importance_avg, perm_importance_avg, r2_global, r2_adj_global)

In [4]:
def nested_cv_flag_bad_subjects(data_, best_features, y_col="Age", diag_col="anydem"):

    # Variables
    y = data_[y_col]
    X_selected = data_[best_features]

    kf = KFold(n_splits=10, shuffle=True, random_state=42)

    param_grid = {
        "model__max_iter": [50, 100, 200],
        "model__max_depth": [3, 5, 7],
        "model__learning_rate": [0.01, 0.05, 0.1]
    }

    r2_scores = []
    y_true_all, y_pred_all = [], []
    results_labels_df = pd.DataFrame(columns=['y_labels', 'y_pred', 'GAP', 'GAP_corrected', 'ID'])
    suspected_bad_subjects = []

    # Nested CV
    for fold, (train_idx, test_idx) in enumerate(kf.split(X_selected)):
        print(f" Fold {fold + 1} (Nested CV)")

        X_train, X_test = X_selected.iloc[train_idx], X_selected.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        pipeline = Pipeline([
            ("scaler", MinMaxScaler((0.05, 0.95))),
            ("model", HistGradientBoostingRegressor(random_state=42))
        ])

        grid_search = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grid,
            scoring="r2",
            cv=5,
            n_jobs=-1,
            verbose=0
        )

        grid_search.fit(X_train, y_train)
        best_model = grid_search.best_estimator_

        y_pred = best_model.predict(X_test)
        r2 = r2_score(y_test, y_pred)
        r2_scores.append(r2)

        print(f" R2 fold {fold + 1}: {r2:.4f}")

        y_true_all.extend(y_test)
        y_pred_all.extend(y_pred)

        gap_test = y_pred - y_test
        gap_train = best_model.predict(X_train) - y_train
        slope, intercept, _, _, _ = linregress(y_train, gap_train)
        corrected_gap = gap_test - (slope * y_test + intercept)

        result = np.column_stack((y_test, y_pred, gap_test, corrected_gap))
        temp_df = pd.DataFrame(result, columns=['y_labels', 'y_pred', 'GAP', 'GAP_corrected'])
        temp_df['ID'] = X_test.index

        results_labels_df = pd.concat([results_labels_df, temp_df], ignore_index=True)

        # Identify subjects that greatly reduce R2 in this fold
        if r2 < 0.25:
            individual_errors = np.abs(y_test - y_pred)
            threshold = np.percentile(individual_errors, 95)  # top 5% worst
            bad_subjects_fold = X_test.index[individual_errors >= threshold].tolist()
            suspected_bad_subjects.extend(bad_subjects_fold)

    # Filter only those that are NOT CN
    non_cn_bad_subjects = [
        subj for subj in set(suspected_bad_subjects)
        if data_.loc[subj, diag_col] != 0
    ]

    print(f" Identified subjects (NOT CN) with negative impact: {len(non_cn_bad_subjects)}")

    # Create new dataframe without those NOT CN subjects
    data_filtered = data_.drop(index=non_cn_bad_subjects)
    print(f" New filtered dataframe: {data_filtered.shape}")

    # Final results
    print("\n Average R2:", np.mean(r2_scores), np.std(r2_scores))
    print(" Global R2:", r2_score(y_true_all, y_pred_all))

    return (data_filtered, non_cn_bad_subjects, r2_scores, results_labels_df)

## All vars model

### Load data

In [5]:
data = pd.read_parquet('../../Data/data_wave1.parquet')

In [6]:
vars_list = ['Family_dementia_any', 'Sex_1F_2M', 'Education', 'Disability', 'Assets', 'Ataxia', 'Bradykinesia', 'Hypertension_1Y_0N', 'Heart_Disease_1Y_0N', 'Vision_problems', 'Audition_problems', 'Cogscore', 'CERADimed', 'CERADrecall', 'DSMIV', 'ICD10', 'Euro-D', 'pulse_pressure', 'orthostatic_drop', 'Stroke_1Y_0N', 'TIA_1Y_0N', 'Back_diseases', 'SRQ', 'NPI distress score', 'NPI severity score', 'Arthritis', 'Cough', 'Breathlessness', 'Breath_problems', 'Angina', 'Stomach_problems', 'Faints', 'Paralysis', 'Limiting_illnesses', 'Skin_disorder', 'Pain', 'Emotional_Disability', 'Physical_activities', 'Walk_1Y_0N', 'Exercise_increase', 'Medic_visits', 'Medications_1Y_0N']

In [7]:
Counter(data.Countries)

Counter({'Cuba': 2923,
         'Puerto Rico': 1992,
         'DR': 1982,
         'Venezuela': 1877,
         'Peru (urban)': 1373,
         'Mexico (urban)': 999,
         'Mexico (rural)': 998,
         'Peru (rural)': 549})

### SFS all subjects

In [8]:
best_features = get_best_features_sfs(data, vars_list,
                                      target_col="Age",
                                      pkl_path="../../SFS/best_features-all-vars_all-subjects.pkl",
                                      force=False)


Loaded best_features from pickle: ['Family_dementia_any', 'Sex_1F_2M', 'Education', 'Disability', 'Assets', 'Ataxia', 'Bradykinesia', 'Hypertension_1Y_0N', 'Heart_Disease_1Y_0N', 'Vision_problems', 'Audition_problems', 'Cogscore', 'CERADrecall', 'DSMIV', 'ICD10', 'Euro-D', 'pulse_pressure', 'orthostatic_drop', 'Stroke_1Y_0N', 'Back_diseases', 'SRQ', 'NPI severity score', 'Arthritis', 'Breathlessness', 'Breath_problems', 'Angina', 'Stomach_problems', 'Faints', 'Paralysis', 'Skin_disorder', 'Pain', 'Emotional_Disability', 'Physical_activities', 'Walk_1Y_0N', 'Exercise_increase', 'Medications_1Y_0N']


In [9]:
## Ensure these are taken into account
best_features.append('Sex_1F_2M')
best_features.append('Education')

best_features = list(np.unique(best_features))

### BBAGs model-all subjects

In [10]:
r2_scores, r2_adj_scores, results_df, shap_imp, perm_imp, r2_global, r2_adj_global = run_nested_cv_hgbr(
    data_=data,
    best_features=best_features,
    y_col="Age",
    diag_col="anydem"
)

 Fold 1 (Nested CV)
 R2 fold 1: 0.2984 | R2 adj: 0.2779 (best hyperparameters: {'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__max_iter': 400})
 Fold 2 (Nested CV)
 R2 fold 2: 0.2894 | R2 adj: 0.2687 (best hyperparameters: {'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__max_iter': 300})
 Fold 3 (Nested CV)
 R2 fold 3: 0.2828 | R2 adj: 0.2619 (best hyperparameters: {'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__max_iter': 300})
 Fold 4 (Nested CV)
 R2 fold 4: 0.3165 | R2 adj: 0.2966 (best hyperparameters: {'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__max_iter': 400})
 Fold 5 (Nested CV)
 R2 fold 5: 0.3342 | R2 adj: 0.3147 (best hyperparameters: {'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__max_iter': 400})
 Fold 6 (Nested CV)
 R2 fold 6: 0.2756 | R2 adj: 0.2545 (best hyperparameters: {'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__max_iter': 400})
 Fold 7 (Nested CV)
 R2 fold 7: 0.3250 | R2 adj: 0.3

In [11]:
results_df

,y_labels,y_pred,GAP,GAP_corrected,ID
0,81.0,80.383340,-0.616660,4.428083,8
1,78.0,72.458613,-5.541387,-2.660037,14
2,73.0,66.384427,-6.615573,-7.339879,19
3,78.0,70.685174,-7.314826,-4.433476,31
4,73.0,79.736234,6.736234,6.011929,33
...,...,...,...,...,...
12688,69.0,79.947514,10.947514,7.270547,12632
12689,84.0,73.950533,-10.049467,-2.665349,12668
12690,69.0,75.930241,6.930241,3.253274,12669
12691,82.0,81.382409,-0.617591,5.291715,12674


In [12]:
results_df['y_pred_corrected'] = results_df['y_labels'] + results_df['GAP_corrected']
results_df = results_df.set_index('ID')

results_df = results_df.sort_index()

df_final = pd.concat([data[['uid','date', 'Age', 'anydem', 'dsmcase', 'cogcase', 'mci', 'Countries'] + best_features], results_df], axis = 1)

In [13]:
df_final.to_parquet('../../Results/all-vars_all-subjects.parquet')

perm_imp.to_frame(name="perm_importance").to_parquet("../../Results/all-vars_all-subjects-perm_imp.parquet")


import pickle

out = {
    "r2_scores": r2_scores,
    "r2_adj_scores": r2_adj_scores,
    "r2_global": r2_global,
    "r2_adj_global": r2_adj_global
}

with open("../../Results/r2_all-vars_all-subjects.pkl", "wb") as f:
    pickle.dump(out, f)
